# Cours — Feature engineering, réglage des modèles et apprentissage non supervisé

**Semaine 4 · Jour 3** · Propilote-Learning

**Au programme**

1. Clustering avec K-Means et clustering hiérarchique
2. Réduction de dimension avec l'analyse en composantes principales (PCA)
3. Feature Engineering et autres modèles de classification
4. Hyperparameter Tuning avec le dataset Titanic

> **Comment utiliser ce notebook**
>
> 1. **Fichier → Enregistrer une copie dans Drive** pour travailler sur ta propre copie.
> 2. Lis chaque leçon et **exécute les cellules de code** (Maj + Entrée) : modifie-les, teste, observe.
> 3. Les lignes `# → …` indiquent le résultat attendu.
> 4. Les cellules **✍️ Essaie ici** sont pour toi : fais les mini-exercices du cours.
> 5. Certaines cellules montrent **volontairement une erreur** (le texte autour l'explique) : lis le message, c'est le but.
> 6. Les exercices notés de la journée sont dans les notebooks d'exercices (liens en bas).

# 1. Clustering avec K-Means et clustering hiérarchique

*Apprentissage non supervisé : regrouper des données sans étiquettes*

**Objectifs**

- Comprendre les bases de l'apprentissage non supervisé et du clustering
- Savoir comment fonctionne K-Means et l'implémenter
- Savoir comment fonctionne le clustering hiérarchique et l'implémenter
- Distinguer K-Means et clustering hiérarchique
- Évaluer et choisir la meilleure méthode de clustering pour ses données

## Introduction à l'apprentissage non supervisé

Jusqu'ici, nous nous sommes concentrés sur des techniques d'apprentissage supervisé, comme la classification et la régression. Ces méthodes s'appuient sur des données étiquetées, où l'objectif est de prédire une variable cible. Dans le monde réel, on rencontre souvent des situations où les données ne sont pas étiquetées. Dans ce cas, l'apprentissage non supervisé permet de trouver des motifs, des relations et des regroupements dans les données sans étiquettes prédéfinies.

- Clustering : regrouper des points de données similaires.
- Réduction de dimension : réduire le nombre de features tout en préservant l'information importante (ex. PCA, t-SNE).
- Association : trouver des associations ou des règles (ex. analyse du panier d'achat).

## Clustering avec K-Means

K-Means est un algorithme de clustering dont l'objectif est de regrouper des points de données similaires. Il est utilisé en apprentissage non supervisé car il n'a pas besoin de données étiquetées (il n'y a pas de « bonne réponse » donnée à l'avance comme en apprentissage supervisé). L'algorithme essaie de trouver seul des motifs dans les données.

Le K de K-Means désigne le nombre de clusters que l'on souhaite créer dans les données. L'algorithme regroupe les points de données en K clusters selon leurs similarités.

#### Comment fonctionne K-Means ?

- Initialisation : on choisit K centroïdes initiaux (points de données choisis aléatoirement, ou via une méthode plus intelligente comme K-Means++). Ces centroïdes représentent le centre de chaque cluster.
- Assignation : chaque point de donnée est assigné au centroïde le plus proche, selon une métrique de distance (généralement la distance euclidienne).
- Mise à jour : les centroïdes sont recalculés en prenant la moyenne de tous les points assignés à ce cluster.
- Répétition : on répète l'assignation et la mise à jour jusqu'à ce que les centroïdes ne changent plus (convergence).

Exemple : imagine un ensemble de points représentant différents fruits, comme des pommes et des oranges. Si tu veux les diviser en 2 groupes (K=2), K-Means trouvera 2 centres — un pour les pommes, un pour les oranges — et assignera chaque fruit au centre le plus proche.

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

X, _ = make_blobs(n_samples=300, centers=4, random_state=42)

kmeans = KMeans(n_clusters=4)
kmeans.fit(X)

plt.scatter(X[:, 0], X[:, 1], c=kmeans.labels_, cmap='viridis')
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], marker='X', s=200, c='red')
plt.show()

## Choisir le nombre de clusters (K)

Choisir le bon nombre de clusters K est essentiel pour obtenir un clustering pertinent et efficace. Si K est trop petit, des groupes distincts risquent d'être fusionnés à tort. Si K est trop grand, les données sont sur-segmentées, ce qui ajoute une complexité inutile.

#### La méthode du coude (Elbow Method)

- Exécute K-Means pour différentes valeurs de K (ex. de 1 à 10).
- Calcule l'inertie (aussi appelée within-cluster sum of squares, WCSS) pour chaque K.
- Trace K en abscisse et l'inertie en ordonnée.
- Cherche le « point de coude » où la baisse de l'inertie ralentit nettement : c'est généralement le K optimal.

Pourquoi ça marche : quand K augmente, l'inertie diminue toujours, car plus de clusters signifie que chaque point est plus proche de son centroïde assigné. Mais au-delà d'un certain point, ajouter des clusters ne réduit plus significativement l'inertie — c'est ce point de ralentissement qui indique le K optimal.

#### Le score de silhouette

Le score de silhouette évalue à quel point les clusters sont bien séparés les uns des autres.

In [ ]:
S = (b - a) / max(a, b)

# a : distance moyenne entre un point et les autres points de son propre cluster
# b : distance moyenne entre un point et les points du cluster voisin le plus proche

- S proche de 1 : bon clustering (les points sont bien à l'intérieur de leur cluster).
- S proche de 0 : clusters qui se chevauchent (points proches de la frontière).
- S proche de -1 : mauvais clustering (points probablement mal assignés).

> 💡 **Astuce** : La méthode du coude est simple et très utilisée mais nécessite une interprétation visuelle. Le score de silhouette donne une métrique plus objective, mais est plus coûteux à calculer.

## Clustering hiérarchique

### Principe

Le clustering hiérarchique construit une hiérarchie de clusters sous forme d'arbre, appelé dendrogramme. Contrairement à K-Means, il n'est pas nécessaire de préciser le nombre de clusters à l'avance.

- Agglomératif (ascendant) : commence avec chaque point de donnée comme son propre cluster, puis fusionne itérativement les clusters les plus proches.
- Divisif (descendant) : commence avec tous les points dans un seul cluster, puis le divise récursivement jusqu'à ce que chaque point soit son propre cluster.

La façon de mesurer la distance entre clusters s'appelle le linkage :

- Single linkage : distance entre les points les plus proches des deux clusters.
- Complete linkage : distance entre les points les plus éloignés des deux clusters.
- Average linkage : distance moyenne entre tous les points des deux clusters.
- Ward : minimise la variance à l'intérieur des clusters.

### Exemple : clustering agglomératif

In [ ]:
from sklearn.cluster import AgglomerativeClustering

agg_clust = AgglomerativeClustering(n_clusters=4, linkage='ward')
agg_clust.fit(X)

plt.scatter(X[:, 0], X[:, 1], c=agg_clust.labels_, cmap='viridis')
plt.show()

Le clustering agglomératif fonctionne bien pour des clusters non sphériques, contrairement à K-Means qui suppose des clusters globulaires. Il ne nécessite pas de centroïdes.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

Z = linkage(X, 'ward')
dendrogram(Z)
plt.show()

## Évaluer la performance d'un clustering

Le clustering est une technique non supervisée : on ne dispose pas d'étiquettes prédéfinies. On a néanmoins besoin de métriques pour évaluer la qualité du regroupement obtenu.

#### Métriques internes (sans vérité terrain)

- Score de silhouette : mesure à quel point les clusters sont séparés (voir plus haut).
- Davies-Bouldin Index (DBI) : évalue la compacité et la séparation des clusters. Plus le DBI est faible, meilleur est le clustering.

#### Métriques externes (avec vérité terrain disponible)

- Adjusted Rand Index (ARI) : mesure la similarité entre les clusters prédits et les vraies étiquettes, en tenant compte du hasard. ARI=1 → clustering parfait, ARI=0 → clustering aléatoire, ARI<0 → pire qu'aléatoire.
- Normalized Mutual Information (NMI) : mesure la quantité d'information partagée entre le clustering et les vraies étiquettes. NMI=1 → correspondance parfaite, NMI=0 → indépendance totale.

> ⚠️ **Attention** : Le clustering est un outil essentiel de l'apprentissage non supervisé, mais chaque algorithme a ses forces et faiblesses : K-Means est efficace sur de gros volumes mais nécessite de fixer K et suppose des clusters sphériques ; le clustering hiérarchique ne nécessite pas de fixer le nombre de clusters à l'avance, mais est plus coûteux en calcul sur de très grands jeux de données.

# 2. Réduction de dimension avec l'analyse en composantes principales (PCA)

*Comprendre et implémenter la technique de réduction de dimension la plus utilisée*

**Objectifs**

- Comprendre les bases de la réduction de dimension et son intérêt
- Comprendre le fonctionnement de la PCA et les mathématiques sous-jacentes
- Implémenter la PCA en Python
- Interpréter les résultats d'une PCA
- Connaître les applications pratiques de la PCA

## Pourquoi réduire la dimension ?

La réduction de dimension consiste à réduire le nombre de features (ou dimensions) d'un jeu de données tout en préservant un maximum d'information. Les données de haute dimension peuvent être difficiles à manipuler à cause du « fléau de la dimension » (curse of dimensionality), qui recouvre plusieurs phénomènes :

- Complexité de calcul accrue.
- Difficulté à visualiser et interpréter les données.
- Éparpillement des points de données (sparsity), rendant plus difficile la détection de motifs pertinents.

Des techniques comme la PCA aident à surmonter ces difficultés en transformant les données dans un espace de dimension réduite, tout en conservant l'essentiel de la variance des données.

## Qu'est-ce que la PCA ?

La PCA (Principal Component Analysis) est une technique linéaire de réduction de dimension qui projette des données de haute dimension sur un sous-espace de dimension inférieure. L'objectif est de trouver les directions (appelées composantes principales) dans lesquelles les données varient le plus. Ces composantes principales sont orthogonales (non corrélées) et ordonnées selon la quantité de variance qu'elles expliquent.

- Composantes principales : les nouveaux axes de l'espace transformé. La première composante principale (PC1) explique le plus de variance, la seconde (PC2) la deuxième plus grande part, etc.
- Variance expliquée : la PCA cherche à maximiser la variance capturée par chaque composante principale.
- Orthogonalité : les composantes principales sont non corrélées, elles capturent des aspects différents des données.

## Comment fonctionne la PCA

- Standardiser les données : la PCA est sensible à l'échelle des données, il faut donc standardiser les features (moyenne = 0, variance = 1).
- Calculer la matrice de covariance : elle capture les relations entre les features.
- Décomposition en valeurs propres : on décompose la matrice de covariance pour trouver ses valeurs propres et vecteurs propres.
- Trier les valeurs propres : on trie les valeurs propres par ordre décroissant et on sélectionne les vecteurs propres correspondants — ce sont les composantes principales.
- Projeter les données : on transforme les données originales dans le nouveau sous-espace en les multipliant par les vecteurs propres sélectionnés.

Exemple : imagine un jeu de données avec deux features — taille et poids. La PCA trouvera un nouvel axe (composante principale) qui capture le plus de variance dans les données. Ce nouvel axe pourrait représenter une combinaison de taille et de poids, comme une « corpulence générale ».

## Implémenter la PCA en Python

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

X, _ = make_blobs(n_samples=300, n_features=5, random_state=42)

# Standardisation des données
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Application de la PCA
pca = PCA(n_components=2)  # Réduction à 2 dimensions
X_pca = pca.fit_transform(X_scaled)

plt.scatter(X_pca[:, 0], X_pca[:, 1])
plt.xlabel('Composante principale 1')
plt.ylabel('Composante principale 2')
plt.title('PCA sur des données d\'exemple')
plt.show()

print("Ratio de variance expliquée :", pca.explained_variance_ratio_)

n_components précise le nombre de composantes principales à conserver. explained_variance_ratio_ indique la proportion de variance expliquée par chaque composante.

#### Choisir le nombre de composantes

- Utilise un scree plot pour visualiser la variance expliquée par chaque composante.
- Conserve assez de composantes pour expliquer un pourcentage souhaité de la variance totale (ex. 95%).

In [ ]:
explained_variance = pca.explained_variance_ratio_
plt.plot(range(1, len(explained_variance) + 1), explained_variance, marker='o')
plt.xlabel('Composante principale')
plt.ylabel('Ratio de variance expliquée')
plt.title('Scree Plot')
plt.show()

## Applications et limites

#### Applications

- Visualisation de données : réduire les données à 2 ou 3 dimensions pour les visualiser facilement.
- Réduction de bruit : retirer les composantes les moins importantes pour se concentrer sur les motifs les plus significatifs.
- Feature engineering : créer de nouvelles features (les composantes principales) pour des modèles de ML.
- Compression d'image : réduire la dimensionnalité des données d'image tout en préservant les features importantes.

#### Limites

- Hypothèse de linéarité : la PCA suppose que les données se situent sur un sous-espace linéaire. Pour des données non linéaires, des techniques comme t-SNE ou UMAP sont plus adaptées.
- Interprétabilité : les composantes principales sont des combinaisons linéaires des features d'origine, ce qui les rend plus difficiles à interpréter.
- Perte d'information : réduire les dimensions implique toujours une perte d'information, même minime.

> 💡 **Astuce** : La PCA reste un outil polyvalent et largement utilisé en Machine Learning et en analyse de données, malgré ses limites.

# 3. Feature Engineering et autres modèles de classification

*Imputation, outliers, one-hot encoding, Gradient Boosting, XGBoost, Random Forest, SVC*

**Objectifs**

- Comprendre le Feature Engineering et savoir l'appliquer
- Découvrir d'autres modèles de classification : XGBoost, Random Forest, SVC, Gradient Boosting
- Comprendre le rôle des hyperparamètres dans ces modèles

## Qu'est-ce que le Feature Engineering ?

Le Feature Engineering, c'est un peu donner des « super-pouvoirs » à un modèle de Machine Learning. C'est le processus qui consiste à créer de nouvelles features pertinentes à partir des données existantes, pour aider le modèle à mieux comprendre les motifs.

Exemple : pour un modèle de prédiction du prix des maisons, au lieu d'utiliser seulement le nombre de chambres, on peut construire une nouvelle feature « surface totale » en additionnant la surface de toutes les pièces. Cela aide le modèle à faire des prédictions plus précises.

De bonnes features peuvent faire qu'un modèle simple surpasse un modèle complexe alimenté par de mauvaises features. C'est comme donner à un détective les bons indices pour résoudre une enquête plus vite : sans bonnes features, les modèles ne peuvent pas voir les motifs cachés dans les données.

## Techniques de Feature Engineering

### Imputation des valeurs manquantes

Le choix de la méthode d'imputation dépend du mécanisme de données manquantes (MCAR/MAR/MNAR) et du type de donnée. Évite le remplissage systématique par zéro.

- SimpleImputer (moyenne/médiane pour le numérique ; valeur la plus fréquente/constante pour le catégoriel).
- KNNImputer pour exploiter la structure locale des données.
- IterativeImputer (MICE) pour les relations multivariées.

> ⚠️ **Attention** : Place toujours l'imputation à l'intérieur d'un Pipeline ajusté uniquement sur le split d'entraînement, pour éviter toute fuite de données (data leakage).

### Gestion des outliers

- Suppression : on retire les lignes contenant des outliers — risqué si les outliers touchent de nombreuses variables (on perdrait trop de données).
- Remplacement : traiter les outliers comme des valeurs manquantes et les imputer.
- Capping : remplacer les valeurs extrêmes par une valeur seuil arbitraire ou issue de la distribution.
- Discrétisation : convertir des variables continues en intervalles (bins) discrets.

### One-hot encoding

Les données catégorielles regroupent des catégories discrètes (couleurs, villes, types de voiture…). Le one-hot encoding transforme chaque catégorie unique en une colonne binaire : un « 1 » dans la colonne de sa catégorie, un « 0 » partout ailleurs.

On l'utilise car de nombreux algorithmes de ML (régression linéaire, arbres de décision…) exigent des entrées numériques. Il évite aussi d'introduire une fausse relation d'ordre entre les catégories, contrairement à un simple encodage par label.

In [ ]:
import pandas as pd

data = {'Person': [1, 2, 3],
        'Favorite Car Color': ['Red', 'Blue', 'Green']}
df = pd.DataFrame(data)

df_encoded = pd.get_dummies(df, columns=['Favorite Car Color'], prefix=['Color'])
print(df_encoded)

> ✏️ **À toi**
>
> Quel est l'objectif principal du one-hot encoding en Machine Learning ? a) Réduire la dimensionnalité des données. b) Représenter des variables catégorielles à plusieurs catégories avec des colonnes binaires. c) Effectuer des opérations arithmétiques sur des variables catégorielles. d) Créer des variables continues à partir de variables catégorielles.

In [ ]:
# ✍️ Essaie ici


## Autres modèles de classification

### Gradient Boosting

Le Gradient Boosting fonctionne comme un processus itératif : tu pars d'un modèle simple peu précis, tu identifies ses erreurs, tu crées un nouveau modèle qui se concentre sur la correction de ces erreurs, puis tu combines les deux modèles pour obtenir une meilleure prédiction. Tu répètes ce processus plusieurs fois.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier()
gb_model.fit(X_train, y_train)

gb_y_pred = gb_model.predict(X_test)
gb_accuracy = accuracy_score(y_test, gb_y_pred)
print(f'Accuracy (Gradient Boosting): {gb_accuracy}')

### XGBoost (Extreme Gradient Boosting)

XGBoost est une version « boostée » et optimisée du Gradient Boosting classique : mêmes principes, avec des astuces d'optimisation supplémentaires qui le rendent plus rapide et plus précis. C'est un choix très populaire dans les compétitions Kaggle, notamment efficace sur des données déséquilibrées.

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')

### Random Forest

Imagine que tu doives prendre une grande décision : au lieu de demander l'avis d'une seule personne, tu demandes à toute une « forêt » de personnes, puis tu tranches à la majorité. En Machine Learning, chaque « arbre » est un modèle de décision simple, entraîné sur un sous-ensemble différent des données et des features. En combinant les prédictions de tous les arbres (vote majoritaire pour la classification), on obtient une prédiction plus précise et fiable.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier()
rf_model.fit(X_train, y_train)

rf_y_pred = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_y_pred)
print(f'Accuracy (Random Forest): {rf_accuracy}')

### Support Vector Classifier (SVC)

Imagine que tu doives tracer une ligne (ou un plan en plus de dimensions) qui sépare des zones sûres de zones dangereuses, en maximisant la distance entre la ligne et les points les plus proches. Un SVM apprend un hyperplan de décision qui maximise cette marge entre les classes.

In [ ]:
from sklearn.svm import SVC

svc_model = SVC()
svc_model.fit(X_train, y_train)

svc_y_pred = svc_model.predict(X_test)
svc_accuracy = accuracy_score(y_test, svc_y_pred)
print(f'Accuracy (SVC): {svc_accuracy}')

## Hyperparamètres des modèles

Le hyperparameter tuning est le processus d'optimisation des hyperparamètres d'un modèle de ML pour obtenir la meilleure performance possible sur une tâche donnée. Les hyperparamètres sont des réglages fixés avant l'entraînement (contrairement aux paramètres appris à partir des données).

#### Hyperparamètres du SVM

- C (paramètre de régularisation) : un C élevé pénalise fortement les erreurs (frontière plus stricte, plus proche des points), un C faible tolère plus d'erreurs pour une marge plus large.
- Kernel : détermine le type de « vision » sur les données — linéaire, polynomial, RBF, etc.
- Gamma : contrôle l'influence de chaque point sur la forme de la frontière de décision (surtout pertinent avec le noyau RBF). Un gamma élevé donne une frontière très sensible aux points individuels ; un gamma faible donne une frontière plus lisse.

#### Hyperparamètres de XGBoost

- learning_rate : la taille du pas d'apprentissage à chaque itération. Plus petit = convergence plus lente mais moins de risque de surapprentissage.
- n_estimators : le nombre d'arbres de boosting entraînés.
- max_depth : la profondeur maximale de chaque arbre. Plus grand = motifs plus complexes captés, mais plus de risque de surapprentissage.
- min_child_weight : la somme minimale de poids requise dans un nœud enfant — un seuil plus élevé prévient le surapprentissage.
- subsample : le pourcentage de lignes utilisées pour construire chaque arbre.

# 4. Hyperparameter Tuning avec le dataset Titanic

*GridSearchCV et RandomizedSearchCV appliqués à un vrai jeu de données*

**Objectifs**

- Comprendre GridSearchCV et RandomizedSearchCV
- Appliquer le hyperparameter tuning à un modèle SVM sur des données réelles
- Comparer performance avec et sans réglage des hyperparamètres

## Préparation des données

On utilise les données de la compétition Titanic: Machine Learning from Disaster.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn import svm

train_data = pd.read_csv('/kaggle/input/titanic/train.csv')

# Retirer les lignes où la cible est manquante
train_data.dropna(axis=0, subset=['Survived'], inplace=True)
y = train_data.Survived
train_data.drop(['Survived'], axis=1, inplace=True)

train_data.drop(['Age'], axis=1, inplace=True)  # colonne retirée pour simplifier

# On ne garde que les colonnes numériques
numeric_cols = [c for c in train_data.columns if train_data[c].dtype in ['int64', 'float64']]
X = train_data[numeric_cols].copy()

print("Shape:", X.shape, y.shape)

Le jeu de données final comporte 5 features et 891 exemples d'entraînement. L'objectif est de prédire quels passagers ont survécu au naufrage du Titanic, à partir de l'identifiant du passager, la classe du billet, le nombre de frères/sœurs ou conjoints à bord, le nombre de parents/enfants à bord, et le prix du billet.

## Score du modèle sans réglage des hyperparamètres

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = svm.SVC()
clf.fit(X_train, y_train)
print('Score avec paramètres par défaut :', clf.score(X_test, y_test))

**Sortie :**

```text
Score avec paramètres par défaut : 0.60
```

> ⚠️ **Attention** : Sans réglage des hyperparamètres, on obtient seulement 60% d'accuracy environ — un score très moyen pour ce problème.

## Réglage avec GridSearchCV

GridSearchCV entraîne le modèle sur toutes les combinaisons possibles des hyperparamètres définis dans une grille — c'est une approche exhaustive, dite « brute force ».

In [ ]:
parameters = {
    'C': [0.01, 1, 5],
    'kernel': ('linear', 'rbf'),
    'gamma': ('scale', 'auto'),
}

gsc = GridSearchCV(estimator=svm.SVC(), param_grid=parameters, cv=5, verbose=1)
gsc.fit(X_train, y_train)

print(f'Meilleurs hyperparamètres : {gsc.best_params_}')
print(f'Meilleur score : {gsc.best_score_}')

gsc_result = pd.DataFrame(gsc.cv_results_).sort_values('mean_test_score', ascending=False)
gsc_result[['param_C', 'param_kernel', 'param_gamma', 'mean_test_score']]

> ⚠️ **Attention** : GridSearchCV explore toutes les combinaisons intermédiaires d'hyperparamètres, ce qui le rend très coûteux en calcul dès que la grille grandit.

## Réglage avec RandomizedSearchCV

Contrairement à GridSearchCV, RandomizedSearchCV sélectionne des valeurs au hasard plutôt que de tester un ensemble prédéterminé de combinaisons. À chaque itération, il essaie une combinaison différente d'hyperparamètres et enregistre la performance du modèle. Après plusieurs itérations, il retourne la meilleure combinaison trouvée — ce qui réduit le calcul inutile.

In [ ]:
# n_iter=5 : au lieu de tester les 12 combinaisons possibles, on en teste 5 au hasard par fold
rsc = RandomizedSearchCV(estimator=svm.SVC(), param_distributions=parameters, n_iter=5, cv=5)
rsc.fit(X_train, y_train)

print(f'Meilleurs hyperparamètres : {rsc.best_params_}')
print(f'Meilleur score : {rsc.best_score_}')

> 💡 **Astuce** : Comme le montrent ces résultats, le réglage des hyperparamètres permet de trouver de meilleurs réglages et d'améliorer la performance du modèle. GridSearchCV, plus lent et plus lourd, est peu adapté aux gros jeux de données. RandomizedSearchCV est un choix naturel pour les gros volumes, mais ne garantit pas de trouver les meilleurs hyperparamètres puisqu'il n'explore qu'un échantillon des combinaisons possibles.

---
# Exercices de la journée
Ouvre chaque notebook d'exercices dans Colab, puis rends-le via GitHub sur Propilote-Learning :

- [Exercises XP : prédiction de maladie cardiaque](https://colab.research.google.com/github/franckkohou/propilote-notebooks/blob/main/semaine-4/jour-3/exercices-xp-maladie-cardiaque.ipynb)
- [Exercises XP Gold : projet complet sur le dataset Titanic](https://colab.research.google.com/github/franckkohou/propilote-notebooks/blob/main/semaine-4/jour-3/exercices-xp-gold-titanic-complet.ipynb)
- [Exercises XP Ninja : construire un CNN simple avec Keras](https://colab.research.google.com/github/franckkohou/propilote-notebooks/blob/main/semaine-4/jour-3/exercices-xp-ninja-cnn-keras.ipynb)
- [Daily Challenge : prédiction du cancer du sein](https://colab.research.google.com/github/franckkohou/propilote-notebooks/blob/main/semaine-4/jour-3/daily-challenge-cancer-du-sein.ipynb)